# Visión: candidatos de balón en los huecos del ATD (Colab, solo inferencia)

Corre `src/ball_detect.py` con **WASB** (Tarashima et al., BMVC 2023; [github.com/nttcom/WASB-SBDT](https://github.com/nttcom/WASB-SBDT), código MIT) y sus **pesos públicos de fútbol** (entrenados por los autores con ISSIA-CNR; nota de licencia en `docs/PLAN.md` §6). **Solo inferencia**, con el preprocesado y el postprocesado de evaluación del propio WASB. Nada se entrena, ajusta ni revisa con el vídeo.

**Reglas (no negociables):**
- **Gemini desactivado** en este notebook (*Configuración → IA de Colab*). No uses el chat ni "explicar error" con datos.
- **No mires las detecciones contra el vídeo** ni dibujes heatmaps o cajas sobre fotogramas: está prohibido verificar la salida con el vídeo.
- Aquí entran solo el vídeo táctico (desde la carpeta compartida de los organizadores) y `vision_bundle.zip` (segmentos + código). La homografía y el ATD se quedan en local.

**Segunda pasada: 4 trozos por fotograma (`--tiles 2`).** La primera pasada (fotograma entero) no pasó el control de `vision_fuse.py`: en 512 × 288 el balón de la cámara táctica queda en ~1 px. Ahora cada fotograma se corta en 2 × 2 trozos de 1024 × 576 que se solapan, y cada uno entra a WASB con su mismo preprocesado: el balón se ve ~1,9× más grande. Un solo intento: si no pasa el control, la visión se descarta.

**Pasos**
1. *Entorno de ejecución → Cambiar tipo → GPU.* Gemini desactivado.
2. **Sube el `vision_bundle.zip` nuevo** (`external/vision/vision_bundle.zip`) a `MyDrive/vision/`, **reemplazando el anterior** (trae `segments_pads.csv` y el `ball_detect.py` con `--tiles`).
3. Celdas 1–3: Drive, bundle, WASB y pesos (los pesos ya están en Drive, no se vuelven a bajar).
4. Celda 4: smoke test con 4 trozos.
5. **Fase 1** (celda 5, ~30–45 min en T4): solo los 6.462 fotogramas de margen, donde se hace el control. Celda 6: descarga `detections.csv` y `status.json` → en local `external/vision/out_tiles/`. **Para aquí** hasta que el control en local diga *check passed*.
6. **Fase 2** (celda 7, varias horas en T4; menos en A100): el resto de los huecos. Reanuda sobre la fase 1, no repite nada. Celda 8: descarga final a la misma carpeta local.


In [ ]:
VIDEO = '/content/drive/MyDrive/BAR-ATH_tactical.mp4'   # acceso directo en la raíz de Mi unidad
WORK = '/content/drive/MyDrive/vision'
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(WORK, exist_ok=True)
assert os.path.exists(VIDEO), 'ajusta VIDEO'
print(os.path.getsize(VIDEO) / 1e9, 'GB')

In [ ]:
import os, shutil
if not os.path.exists(f'{WORK}/vision_bundle.zip'):   # reemplázalo en Drive por el nuevo antes de ejecutar
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), f'{WORK}/vision_bundle.zip')
!rm -rf /content/vision && mkdir -p /content/vision && unzip -q {WORK}/vision_bundle.zip -d /content/vision && ls -R /content/vision
assert os.path.exists('/content/vision/segments_pads.csv'), 'bundle viejo: sube el vision_bundle.zip nuevo'


In [ ]:
# WASB (código MIT) y sus pesos de fútbol, el mismo archivo que descarga su setup_weights.sh
!pip -q install omegaconf gdown
!git clone -q https://github.com/nttcom/WASB-SBDT /content/WASB-SBDT && git -C /content/WASB-SBDT rev-parse HEAD
W = f'{WORK}/wasb_soccer_best.pth.tar'
if not os.path.exists(W):
    !gdown -q 1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l -O {W}
!ls -la {W}

In [ ]:
# Smoke test con 4 trozos: 20 tripletes (22 fotogramas) de punta a punta
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments.csv --wasb /content/WASB-SBDT --weights {W} --tiles 2 --out /content/smoke_tiles --smoke
!cat /content/smoke_tiles/status.json; wc -l /content/smoke_tiles/detections.csv


In [ ]:
# FASE 1: solo los márgenes (donde el ATD tiene balón). Reanudable: vuelve a ejecutar (y la 3) si Colab se desconecta
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments_pads.csv --wasb /content/WASB-SBDT --weights {W} --tiles 2 --out {WORK}/out_tiles 2>&1 | tee -a {WORK}/log_tiles.txt
!cat {WORK}/out_tiles/status.json


In [ ]:
# Descarga de la fase 1 -> en local external/vision/out_tiles/ ; luego: python scripts/vision_fuse.py --det external/vision/out_tiles --check-only
from google.colab import files
for f in ('detections.csv', 'status.json'):
    files.download(f'{WORK}/out_tiles/{f}')


In [ ]:
# FASE 2: solo si el control en local dio 'check passed'. Mismo --out: salta lo hecho en la fase 1
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments.csv --wasb /content/WASB-SBDT --weights {W} --tiles 2 --out {WORK}/out_tiles 2>&1 | tee -a {WORK}/log_tiles.txt
!cat {WORK}/out_tiles/status.json


In [ ]:
# Opcional: déjala en cola tras la celda 7 para cerrar la sesión al terminar y no gastar unidades.
# Luego baja detections.csv y status.json desde Drive (MyDrive/vision/out_tiles/) en el navegador.
import time
!tail -3 {WORK}/log_tiles.txt; cat {WORK}/out_tiles/status.json
from google.colab import drive, runtime
drive.flush_and_unmount()   # espera a que Drive termine de sincronizar los archivos
time.sleep(30)
runtime.unassign()          # cierra la sesión y libera la GPU


In [ ]:
# Descarga final -> en local external/vision/out_tiles/ (reemplaza los de la fase 1)
from google.colab import files
for f in ('detections.csv', 'status.json'):
    files.download(f'{WORK}/out_tiles/{f}')
files.download(f'{WORK}/log_tiles.txt')
